In [1]:
!pip install -q pandas scikit-learn mysql-connector-python sqlalchemy langchain-community langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, IsolationForest

# 1. Load Dataset (Personal_Finance_Dataset.csv or synthetic fallback)
try:
    df = pd.read_csv('Personal_Finance_Dataset.csv')
    df = df.rename(columns={
        'Date': 'date',
        'Transaction Description': 'description',
        'Category': 'category',
        'Amount': 'amount'
    })
    print("Loaded Personal_Finance_Dataset.csv successfully.")
except Exception as e:
    print("Dataset file not found or failed to load. Generating 500 synthetic PKR transactions...")
    np.random.seed(42)
    categories = ['Food & Drink', 'Rent', 'Utilities', 'Shopping', 'Subscriptions']
    vendors = {
        'Food & Drink': ['Foodpanda Order', 'KFC Clifton', 'Imtiaz Supermarket', 'Savour Foods'],
        'Rent': ['Monthly Apartment Rent', 'House Rent Transfer'],
        'Utilities': ['K-Electric Bill Payment', 'SSGC Gas Bill', 'StormFiber Broadband'],
        'Shopping': ['Daraz.pk Online Purchase', 'Gul Ahmed Store', 'Outfitters Store'],
        'Subscriptions': ['Netflix Monthly Subscription', 'Spotify Premium', 'LinkedIn Premium']
    }

    data = []
    for _ in range(500):
        cat = np.random.choice(categories)
        desc = np.random.choice(vendors[cat])
        amt = np.random.uniform(500, 15000) if cat != 'Rent' else np.random.uniform(40000, 80000)
        data.append({
            'date': pd.date_range(start='2024-01-01', end='2024-06-30').sample(1).dt.strftime('%Y-%m-%d').values[0],
            'description': desc,
            'amount': round(amt, 2),
            'category': cat
        })
    df = pd.DataFrame(data)

# Ensure data types are consistent
df['amount'] = df['amount'].astype(float)
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

# 2. Train RandomForest Classifier for Auto-Categorization
vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(df['description'])
y = df['category']

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_text, y)
df['predicted_category'] = clf.predict(X_text)

# 3. Train Isolation Forest for Anomaly Detection (Unusually large amounts)
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['is_anomaly'] = iso_forest.fit_predict(df[['amount']])
df['is_anomaly'] = df['is_anomaly'].apply(lambda x: 1 if x == -1 else 0)

print(f"\nProcessing Complete: {len(df)} records processed.")
print(f"Anomalies detected: {df['is_anomaly'].sum()}")

Loaded Personal_Finance_Dataset.csv successfully.

Processing Complete: 1500 records processed.
Anomalies detected: 75


In [4]:
import mysql.connector
from sqlalchemy import create_engine, text
from google.colab import userdata

# Fetch credentials securely and strip extra spaces
db_host = str(userdata.get('MYSQL_HOST')).strip()
db_port = str(userdata.get('MYSQL_PORT')).strip()
db_user = str(userdata.get('MYSQL_USER')).strip()
db_pass = str(userdata.get('MYSQL_PASSWORD')).strip()
db_name = str(userdata.get('MYSQL_DB')).strip()

# Create SQLAlchemy Connection Engine
engine_url = f"mysql+mysqlconnector://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(engine_url)

# Define 3NF Database Schema
schema_sql = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INT AUTO_INCREMENT PRIMARY KEY,
    category_name VARCHAR(50) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INT AUTO_INCREMENT PRIMARY KEY,
    date DATE NOT NULL,
    description VARCHAR(255) NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    category_id INT,
    is_anomaly TINYINT(1) DEFAULT 0,
    FOREIGN KEY (category_id) REFERENCES categories(category_id) ON DELETE SET NULL
);
"""

# Execute Schema Creation
with engine.begin() as conn:
    for statement in schema_sql.split(';'):
        if statement.strip():
            conn.execute(text(statement))

# Insert Categories
unique_cats = pd.DataFrame({'category_name': df['predicted_category'].unique()})
unique_cats.to_sql('categories', con=engine, if_exists='append', index=False, method='multi')

# Map Category IDs back to Transactions
cat_mapping = pd.read_sql("SELECT category_id, category_name FROM categories", con=engine)
df_mapped = df.merge(cat_mapping, left_on='predicted_category', right_on='category_name')

# Insert Transactions
db_transactions = df_mapped[['date', 'description', 'amount', 'category_id', 'is_anomaly']]
db_transactions.to_sql('transactions', con=engine, if_exists='append', index=False)

print("3NF Database Schema created and data successfully loaded into MySQL.")

3NF Database Schema created and data successfully loaded into MySQL.


In [10]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_groq import ChatGroq
from google.colab import userdata

# Fetch Groq API Key
groq_api_key = str(userdata.get('GROQ_API_KEY')).strip()

# Connect LangChain to MySQL Database Engine
db = SQLDatabase(engine, include_tables=['categories', 'transactions'])
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key
)

# Initialize Agent with parsing error handling
sql_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="zero-shot-react-description",
    handle_parsing_errors=True,  # Fixes OutputParserException automatically
    verbose=True
)

def query_spendwise(question):
    try:
        response = sql_agent.invoke({"input": question})
        return response.get('output', 'No response generated.')
    except Exception as e:
        return f"Database Query Error: {str(e)}"

In [11]:
import gradio as gr
import matplotlib.pyplot as plt

def generate_spending_chart():
    fig, ax = plt.subplots(figsize=(8, 4))
    summary = df.groupby('predicted_category')['amount'].sum().sort_values(ascending=False)
    summary.plot(kind='bar', ax=ax, color='#1f77b4')
    ax.set_title('Total Expenditure by Category')
    ax.set_ylabel('Amount')
    ax.set_xlabel('Category')
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig

with gr.Blocks(theme=gr.themes.Soft(), title="SpendWise AI Platform") as demo:
    gr.Markdown("# 💳 SpendWise — AI Personal Finance Intelligence Platform")

    with gr.Tab("📁 Upload & View Data"):
        gr.Markdown("### Normalized Database Preview (3NF Schema)")
        gr.DataFrame(df[['date', 'description', 'amount', 'predicted_category', 'is_anomaly']].head(25))

    with gr.Tab("📈 Spend Analytics"):
        gr.Markdown("### Category Spending Distribution")
        chart_btn = gr.Button("Generate Chart")
        chart = gr.Plot()
        chart_btn.click(fn=generate_spending_chart, outputs=chart)

    with gr.Tab("🤖 Conversational Assistant (NL-to-SQL)"):
        gr.Markdown("### Query your financial database in plain English")

        chatbot = gr.Chatbot(height=350)
        msg = gr.Textbox(placeholder="Ask e.g., 'What is the total spending on Rent?' or 'List all anomalous transactions.'")
        clear = gr.Button("Clear Chat")

        def user_chat(user_message, history):
            if history is None:
                history = []
            bot_response = query_spendwise(user_message)
            history.append({"role": "user", "content": user_message})
            history.append({"role": "assistant", "content": bot_response})
            return "", history

        msg.submit(user_chat, [msg, chatbot], [msg, chatbot])
        clear.click(lambda: [], None, chatbot, queue=False)

# Launch updated app
demo.launch(share=True, debug=True)

/tmp/ipykernel_1863/2060860131.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="SpendWise AI Platform") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://81521ebe9c0ef25387.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: categories, transactionsWith the list of tables, I can now query the schema of the most relevant tables to understand their structure.

Action: sql_db_schema
Action Input: categories, transactions
CREATE TABLE categories (
	category_id INTEGER NOT NULL AUTO_INCREMENT, 
	category_name VARCHAR(50) NOT NULL, 
	PRIMARY KEY (category_id)
)

/*
3 rows from categories table:
category_id	category_name
7	Entertainment
1	Food & Drink
8	Health & Fitness
*/


CREATE TABLE transactions (
	transaction_id INTEGER NOT NULL AUTO_INCREMENT, 
	date DATE NOT NULL, 
	description VARCHAR(255) NOT NULL, 
	amount DECIMAL(10, 2) NOT NULL, 
	category_id INTEGER, 
	is_anomaly TINYINT(1) DEFAULT '0', 
	PRIMARY KEY (transaction_id), 
	CONSTRAINT transactions_ibfk_1 FOREIGN KEY(category_id) REF